# Assignment 2: Complexity Budgeting for Regression and Optimization

Amanveer Wesley | MSMA 5023 Programming and Algorithmic Thinking for Applied AI

This notebook continues Assignment 1. It keeps the same data, the same grouped split
and the same preprocessing, then measures what four training configurations cost in
runtime and what they deliver in F1 under one fixed compute budget.

## Dataset Description

- **Source:** Default of Credit Card Clients, UCI Machine Learning Repository
  (Yeh, 2009), https://doi.org/10.24432/C55S3H. The raw archive is read from the
  working folder and downloaded from UCI if it is missing.
- **License:** Creative Commons Attribution 4.0 International (CC BY 4.0).
- **Size:** 30,000 rows, 23 predictor features, one ID column and one target column.
- **Target variable:** `default_next_month` (1 = the client defaulted on the next
  payment, 0 = the client did not). About 22% of rows are defaults.
- **Problem type:** Binary classification.
- **Primary metric:** F1 at the default 0.5 threshold. Recall, precision, accuracy
  and ROC-AUC are reported as context.

## Experiment rules

- **Compute budget:** 15 minutes of wall-clock time on one CPU thread for all model
  fitting, tuning, cross-validation and timing work. The ledger at the end reports
  what was used. This is a scaled-down stand-in for the assignment's "90 minutes of
  GPU time" scenario.
- **Four configurations:** (1) default `LogisticRegression()`, (2) SGD with a tuned
  constant learning rate and a fixed epoch budget, (3) SGD with manual early stopping,
  and (4) SGD with feature reduction (`SelectKBest`).
- **No leakage:** preprocessing, feature selection, learning-rate tuning and
  early-stopping decisions use training rows only. The stopping set is carved out of
  the training rows by profile group, so it never overlaps an evaluation fold.
- **Test set:** the Assignment 1 test partition was already scored in Assignment 1.
  It is not loaded into any model or metric here.
- **Selection rule, fixed before the results:** a configuration is eligible if its
  mean cross-validated F1 is within one standard error of the best mean F1. Among
  eligible configurations, the one with the lowest median total training time wins.
- **Timing:** one thread, one untimed warm-up, then the median of repeated runs.

In [ ]:
# For Google Colab: run this cell BEFORE the imports. A fresh runtime is recommended.
import sys
import subprocess
if 'google.colab' in sys.modules:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'numpy==2.2.6', 'pandas==2.2.3', 'scikit-learn==1.6.1',
        'scipy==1.15.3', 'matplotlib==3.10.1', 'xlrd==2.0.1'])

In [ ]:
import os
# One thread for every numerical library. These must be set before NumPy is imported.
for _var in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
    os.environ[_var] = '1'

import hashlib
import io
import json
import platform
import time
import urllib.request
import warnings
import zipfile
from collections import defaultdict
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

threadpool_limits(limits=1)   # also cap any thread pool that is already loaded
NOTEBOOK_START = time.perf_counter()
CPU_START = time.process_time()

# ---- Settings fixed before any result was seen ----
SEED = 42
BUDGET_SECONDS = 15 * 60                 # fixed compute budget: 15 minutes, one thread
EPOCH_BUDGET = 20                        # passes over the training data for SGD
ETA_GRID = [0.0003, 0.001, 0.003, 0.01]  # constant learning rates tried for SGD
ALPHA = 1e-4                             # L2 strength for SGD (scikit-learn default)
PATIENCE = 3                             # early stopping: epochs without improvement
MIN_DELTA = 1e-4                         # early stopping: smallest loss gain that counts
STOP_FRACTION = 0.15                     # share of training rows held out for stopping
K_FEATURES = 30                          # encoded columns kept by SelectKBest
TIMING_REPEATS = 7                       # timed repeats per configuration
STABILITY_SEEDS = [42, 43, 44, 45, 46]   # seeds for the convergence stability check
SCALING_FRACTIONS = [0.25, 0.5, 1.0]
SCALING_REPEATS = 3

SOURCE_URL = 'https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip'
ROOT = Path.cwd()
OUT = ROOT / os.environ.get('A2_OUTPUT_SUBDIR', 'a2_outputs')
OUT.mkdir(parents=True, exist_ok=True)
ARCHIVE = Path(os.environ.get('A1_ARCHIVE', ROOT / 'uci_credit_default.zip'))
SPLIT_CSV = Path(os.environ.get('A1_SPLIT_CSV', ROOT / 'a1_split_ids.csv'))
warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

CONFIGS = ['baseline', 'sgd', 'sgd_early_stop', 'sgd_kbest']
LABELS = {'baseline': 'Baseline (L-BFGS)', 'sgd': 'SGD',
          'sgd_early_stop': 'SGD + early stopping', 'sgd_kbest': 'SGD + feature reduction'}
COLORS = {'baseline': '#2a78d6', 'sgd': '#eb6834', 'sgd_early_stop': '#1baf7a',
          'sgd_kbest': '#eda100'}
INK, MUTED = '#0b0b0b', '#52514e'

# Budget ledger: every block of fitting or benchmarking work adds its seconds here.
ledger = defaultdict(float)


class Spend:
    """Context manager that adds the elapsed seconds of a block to the budget ledger."""

    def __init__(self, stage):
        self.stage = stage

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, *exc):
        ledger[self.stage] += time.perf_counter() - self.start
        return False


print('Python', platform.python_version(), '| scikit-learn', version('scikit-learn'),
      '| numpy', version('numpy'), '| pandas', version('pandas'))
print('Machine:', platform.platform(), '|', platform.processor() or 'processor not reported')

## 1. Data and the Assignment 1 split

The loading code and the categorical handling match Assignment 1. The split file from
Assignment 1 assigns every customer to train, validation or test. Rows with identical
feature profiles share a group, and a group never sits in two partitions.

In [ ]:
if not ARCHIVE.exists():
    urllib.request.urlretrieve(SOURCE_URL, ARCHIVE)
archive_hash = hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
with zipfile.ZipFile(ARCHIVE) as archive:
    xls_name = next(n for n in archive.namelist() if n.endswith('.xls'))
    df = pd.read_excel(io.BytesIO(archive.read(xls_name)), header=1, engine='xlrd')
df = df.rename(columns={df.columns[-1]: 'default_next_month'})
assert df.shape == (30000, 25) and df['ID'].is_unique

FEATURES = [c for c in df if c not in ['ID', 'default_next_month']]
PAY = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
CAT = ['SEX', 'EDUCATION', 'MARRIAGE'] + PAY
NUM = [c for c in FEATURES if c not in CAT]
X = df[FEATURES].copy()
y = df.default_next_month.astype(int)
for col in CAT:
    X[col] = X[col].map(lambda v: str(int(v)) if pd.notna(v) else np.nan)
groups = pd.util.hash_pandas_object(X, index=False).astype(str)

if SPLIT_CSV.exists():
    split = pd.read_csv(SPLIT_CSV)
    assert (split['ID'].to_numpy() == df['ID'].to_numpy()).all()
    partition = split['partition'].to_numpy()
    split_source = f'Assignment 1 split file ({SPLIT_CSV.name})'
else:
    # Same construction as Assignment 1. Copy a1_split_ids.csv next to this notebook
    # when you have it, because a rebuilt split can differ across library versions.
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    trainval_idx, test_idx = next(outer.split(X, y, groups))
    inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED + 1)
    tr_rel, va_rel = next(inner.split(X.iloc[trainval_idx], y.iloc[trainval_idx],
                                      groups.iloc[trainval_idx]))
    partition = np.array(['train'] * len(df), dtype=object)
    partition[trainval_idx[va_rel]] = 'validation'
    partition[test_idx] = 'test'
    split_source = 'rebuilt with the Assignment 1 code (no split file found)'

train_i, val_i, test_i = [np.flatnonzero(partition == p) for p in ['train', 'validation', 'test']]
trainval_i = np.sort(np.concatenate([train_i, val_i]))
assert (len(train_i), len(val_i), len(test_i)) == (18004, 5996, 6000)
for a, b in [(train_i, val_i), (train_i, test_i), (val_i, test_i)]:
    assert set(groups.iloc[a]).isdisjoint(groups.iloc[b]), 'profile groups overlap partitions'

# Only training and validation rows are used from here on. The test rows stay untouched.
Xtr, ytr, gtr = X.iloc[train_i], y.iloc[train_i].to_numpy(), groups.iloc[train_i].to_numpy()
Xva, yva = X.iloc[val_i], y.iloc[val_i].to_numpy()
Xtv, ytv, gtv = X.iloc[trainval_i], y.iloc[trainval_i].to_numpy(), groups.iloc[trainval_i].to_numpy()
folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(Xtv, ytv, gtv))
for fold_train, fold_eval in folds:
    assert set(gtv[fold_train]).isdisjoint(gtv[fold_eval]), 'profile groups overlap CV folds'

print('Split source:', split_source)
print('Rows (train, validation, test):', len(train_i), len(val_i), len(test_i))
print('Default rate (train, validation):', round(ytr.mean(), 4), round(yva.mean(), 4))
print('Leakage checks passed: partitions and CV folds share no profile group.')

## 2. Helpers

Every configuration goes through the same three timed stages: preprocessing, an
optional extra step (feature selection or the stopping-set split) and the model fit.
The three SGD configurations share one training loop built on `partial_fit`, so the
only differences between them are the ones named in their labels.

In [ ]:
def make_preprocessor():
    """The Assignment 1 preprocessor: impute and scale numbers, impute and one-hot codes."""
    return ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')),
                              ('scaler', StandardScaler())]), NUM),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                  ('encoder', OneHotEncoder(handle_unknown='ignore'))]), CAT)
    ], sparse_threshold=1.0)


def log_loss_from_scores(scores, labels):
    """Mean logistic loss from raw decision scores. Stable for large scores."""
    signed = np.where(labels == 1, -scores, scores)
    return float(np.mean(np.logaddexp(0.0, signed)))


def stopping_epoch(losses, patience=PATIENCE, min_delta=MIN_DELTA):
    """Apply the early-stopping rule to a list of per-epoch losses.

    Returns (epochs_run, best_epoch), both counted from 1.
    """
    best_loss, best_epoch, waited = np.inf, 0, 0
    for epoch, loss in enumerate(losses, start=1):
        if loss < best_loss - min_delta:
            best_loss, best_epoch, waited = loss, epoch, 0
        else:
            waited += 1
            if waited >= patience:
                return epoch, best_epoch
    return len(losses), best_epoch


def train_sgd(Z, labels, eta0, seed, max_epochs=EPOCH_BUDGET, Zstop=None, ystop=None,
              early_stop=False, record=False):
    """Train logistic regression with per-sample SGD, one `partial_fit` call per epoch.

    early_stop=True stops when the stopping-set loss has not improved by MIN_DELTA for
    PATIENCE epochs, then restores the weights from the best epoch.
    record=True stores the training loss (and stopping-set loss) after every epoch.
    """
    model = SGDClassifier(loss='log_loss', penalty='l2', alpha=ALPHA,
                          learning_rate='constant', eta0=eta0,
                          random_state=np.random.RandomState(seed))
    history = {'train_loss': [], 'stop_loss': []}
    best_loss, best_epoch, waited = np.inf, 0, 0
    best_coef = best_intercept = None
    epochs_run = 0
    for epoch in range(1, max_epochs + 1):
        model.partial_fit(Z, labels, classes=np.array([0, 1]))
        epochs_run = epoch
        if record:
            history['train_loss'].append(log_loss_from_scores(model.decision_function(Z), labels))
        if Zstop is not None:
            stop_loss = log_loss_from_scores(model.decision_function(Zstop), ystop)
            history['stop_loss'].append(stop_loss)
            if early_stop:
                if stop_loss < best_loss - MIN_DELTA:
                    best_loss, best_epoch, waited = stop_loss, epoch, 0
                    best_coef, best_intercept = model.coef_.copy(), model.intercept_.copy()
                else:
                    waited += 1
                    if waited >= PATIENCE:
                        break
    if early_stop and best_coef is not None:
        model.coef_, model.intercept_ = best_coef, best_intercept
    info = {'epochs_run': epochs_run, 'best_epoch': best_epoch if early_stop else epochs_run,
            'best_stop_loss': best_loss if early_stop else None, 'history': history}
    return model, info


class Fitted:
    """A fitted configuration: preprocessor, optional selector and model."""

    def __init__(self, preprocessor, model, selector=None):
        self.preprocessor, self.model, self.selector = preprocessor, model, selector

    def scores(self, frame):
        Z = self.preprocessor.transform(frame)
        if self.selector is not None:
            Z = self.selector.transform(Z)
        return self.model.decision_function(Z)


def evaluate(fitted, frame, labels):
    scores = fitted.scores(frame)
    predictions = (scores > 0).astype(int)
    return {'f1': float(f1_score(labels, predictions, zero_division=0)),
            'recall': float(recall_score(labels, predictions, zero_division=0)),
            'precision': float(precision_score(labels, predictions, zero_division=0)),
            'accuracy': float(accuracy_score(labels, predictions)),
            'roc_auc': float(roc_auc_score(labels, scores))}


def stop_split(frame, labels, group_ids, seed):
    """Hold out STOP_FRACTION of the training rows by profile group for early stopping."""
    splitter = GroupShuffleSplit(n_splits=1, test_size=STOP_FRACTION, random_state=seed)
    fit_idx, stop_idx = next(splitter.split(frame, labels, group_ids))
    assert set(group_ids[fit_idx]).isdisjoint(group_ids[stop_idx])
    return fit_idx, stop_idx


def fit_config(name, frame, labels, group_ids, eta0, seed):
    """Fit one configuration on raw training rows. Returns (Fitted, times, info).

    times has preprocess, extra, fit and total seconds. `extra` is feature selection
    for sgd_kbest and the stopping-set split for sgd_early_stop.
    """
    times = {'preprocess': 0.0, 'extra': 0.0, 'fit': 0.0}
    info = {'epochs': None, 'n_features': None}
    selector = None
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        if name == 'sgd_early_stop':
            start = time.perf_counter()
            fit_idx, stop_idx = stop_split(frame, labels, group_ids, seed)
            times['extra'] = time.perf_counter() - start
            start = time.perf_counter()
            preprocessor = make_preprocessor()
            Z = preprocessor.fit_transform(frame.iloc[fit_idx])
            Zstop = preprocessor.transform(frame.iloc[stop_idx])
            times['preprocess'] = time.perf_counter() - start
            start = time.perf_counter()
            model, run = train_sgd(Z, labels[fit_idx], eta0, seed, Zstop=Zstop,
                                   ystop=labels[stop_idx], early_stop=True)
            times['fit'] = time.perf_counter() - start
            info.update(epochs=run['epochs_run'], best_epoch=run['best_epoch'],
                        rows_fitted=int(len(fit_idx)))
        else:
            start = time.perf_counter()
            preprocessor = make_preprocessor()
            Z = preprocessor.fit_transform(frame)
            times['preprocess'] = time.perf_counter() - start
            if name == 'sgd_kbest':
                start = time.perf_counter()
                selector = SelectKBest(f_classif, k=min(K_FEATURES, Z.shape[1]))
                Z = selector.fit_transform(Z, labels)
                times['extra'] = time.perf_counter() - start
            start = time.perf_counter()
            if name == 'baseline':
                model = LogisticRegression().fit(Z, labels)
                info['epochs'] = int(np.atleast_1d(model.n_iter_).max())
            else:
                model, run = train_sgd(Z, labels, eta0, seed)
                info['epochs'] = run['epochs_run']
            times['fit'] = time.perf_counter() - start
            info['rows_fitted'] = int(len(labels))
        info['n_features'] = int(Z.shape[1])
    times['total'] = times['preprocess'] + times['extra'] + times['fit']
    return Fitted(preprocessor, model, selector), times, info


def tune_eta(frame, labels, group_ids, seed):
    """Pick the SGD learning rate using training rows only.

    Each rate in ETA_GRID is trained with early stopping on a grouped stopping set that
    is carved out of the training rows. The rate with the lowest stopping-set loss wins.
    """
    fit_idx, stop_idx = stop_split(frame, labels, group_ids, seed)
    preprocessor = make_preprocessor()
    Z = preprocessor.fit_transform(frame.iloc[fit_idx])
    Zstop = preprocessor.transform(frame.iloc[stop_idx])
    table = []
    for eta0 in ETA_GRID:
        _, run = train_sgd(Z, labels[fit_idx], eta0, seed, Zstop=Zstop,
                           ystop=labels[stop_idx], early_stop=True)
        table.append({'eta0': eta0, 'best_stop_loss': run['best_stop_loss'],
                      'best_epoch': run['best_epoch'], 'epochs_run': run['epochs_run']})
    best = min(table, key=lambda row: row['best_stop_loss'])
    return best['eta0'], table


def time_config(name, frame, labels, group_ids, eta0, seed, repeats=TIMING_REPEATS):
    """Median stage times over repeated fits, after one untimed warm-up fit."""
    fit_config(name, frame, labels, group_ids, eta0, seed)
    runs = [fit_config(name, frame, labels, group_ids, eta0, seed)[1] for _ in range(repeats)]
    summary = {key: float(np.median([r[key] for r in runs]))
               for key in ['preprocess', 'extra', 'fit', 'total']}
    summary['total_min'] = float(np.min([r['total'] for r in runs]))
    summary['total_max'] = float(np.max([r['total'] for r in runs]))
    summary['repeats'] = repeats
    return summary

## 3. Baseline: default logistic regression

`LogisticRegression()` with default parameters (L-BFGS solver, L2 penalty, C = 1,
at most 100 iterations). It is trained on the 18,004 training rows and scored on the
5,996 validation rows. Every L-BFGS iteration uses the full training set, so this is
the batch method in the comparison.

In [ ]:
single = {}      # results on the Assignment 1 train and validation split
with Spend('single_split_fits'):
    fitted, times, info = fit_config('baseline', Xtr, ytr, gtr, None, SEED)
    single['baseline'] = {'validation': evaluate(fitted, Xva, yva), 'info': info}
    Z_baseline = fitted.preprocessor.transform(Xtr)
    baseline_train_loss = log_loss_from_scores(fitted.model.decision_function(Z_baseline), ytr)
baseline_converged = info['epochs'] < 100
print('Encoded training matrix:', Z_baseline.shape)
print('L-BFGS iterations:', info['epochs'], '| converged within 100:', baseline_converged)
print('Training log loss:', round(baseline_train_loss, 4))
print('Validation metrics:', {k: round(v, 4) for k, v in single['baseline']['validation'].items()})

## 4. Optimization strategies

Three alternatives to the baseline, each a strategy from the assignment list:

1. **SGD with learning-rate tuning.** Per-sample stochastic gradient descent with a
   constant learning rate picked from a small grid, run for a fixed budget of
   20 epochs. Expected effect: each update costs one row instead of the full set, so
   the model should need fewer passes over the data.
2. **SGD with manual early stopping.** The same model, but 15% of the training rows
   (by profile group) are held out, and training stops when their loss has not
   improved for 3 epochs. Expected effect: fewer epochs, so less fit time.
3. **SGD with feature reduction.** `SelectKBest` keeps the 30 encoded columns with the
   highest ANOVA F score before SGD. Expected effect: a smaller matrix, so cheaper
   epochs, at some cost in F1.

The learning rate is tuned once on the training rows and shared by all three.

In [ ]:
with Spend('tuning'):
    eta_single, eta_table = tune_eta(Xtr, ytr, gtr, SEED)
print('Learning-rate tuning on the training rows (grouped stopping set):')
print(pd.DataFrame(eta_table).round(5).to_string(index=False))
print('Chosen learning rate:', eta_single)

with Spend('single_split_fits'):
    for name in ['sgd', 'sgd_early_stop', 'sgd_kbest']:
        fitted, times, info = fit_config(name, Xtr, ytr, gtr, eta_single, SEED)
        single[name] = {'validation': evaluate(fitted, Xva, yva), 'info': info}
        print(f"{LABELS[name]:<26} epochs={info['epochs']:<3} features={info['n_features']:<3} "
              f"validation F1={single[name]['validation']['f1']:.4f} "
              f"recall={single[name]['validation']['recall']:.4f}")

### Runtime of each configuration

Each configuration is fitted once as a warm-up and then seven more times. The table
reports the median of each stage. Speedup is the baseline total divided by the
configuration total, so a value above 1 means faster than the baseline.

In [ ]:
timing = {}
with Spend('timing_repeats'):
    for name in CONFIGS:
        timing[name] = time_config(name, Xtr, ytr, gtr, eta_single, SEED)

runtime_rows = []
for name in CONFIGS:
    t, info, val = timing[name], single[name]['info'], single[name]['validation']
    runtime_rows.append({
        'configuration': LABELS[name],
        'preprocess_s': t['preprocess'], 'extra_s': t['extra'], 'fit_s': t['fit'],
        'total_s': t['total'], 'passes': info['epochs'],
        'fit_s_per_pass': t['fit'] / info['epochs'],
        'speedup_total': timing['baseline']['total'] / t['total'],
        'speedup_fit': timing['baseline']['fit'] / t['fit'],
        'preprocess_share': t['preprocess'] / t['total'],
        'val_f1': val['f1'], 'val_recall': val['recall'], 'val_accuracy': val['accuracy'],
        'val_f1_diff': val['f1'] - single['baseline']['validation']['f1']})
runtime_table = pd.DataFrame(runtime_rows)
print(runtime_table.round(4).to_string(index=False))

## 5. Convergence

Training loss is recorded after every epoch with `partial_fit`. The SGD curve is
repeated for five seeds to show how stable it is. The stopping-set loss is the curve
the early-stopping rule watches. The baseline curve refits L-BFGS with a growing
iteration cap, which shows how many full passes the batch method needs to reach the
same loss. All curves use the same rows: the training rows minus the stopping set.

In [ ]:
convergence = {'seeds': STABILITY_SEEDS, 'sgd_train_loss': [], 'sgd_stop_loss': [],
               'stop_epochs': [], 'best_epochs': []}
with Spend('convergence'):
    fit_idx, stop_idx = stop_split(Xtr, ytr, gtr, SEED)
    conv_pre = make_preprocessor()
    Zc = conv_pre.fit_transform(Xtr.iloc[fit_idx])
    Zc_stop = conv_pre.transform(Xtr.iloc[stop_idx])
    yc, yc_stop = ytr[fit_idx], ytr[stop_idx]
    for seed in STABILITY_SEEDS:
        _, run = train_sgd(Zc, yc, eta_single, seed, Zstop=Zc_stop, ystop=yc_stop, record=True)
        convergence['sgd_train_loss'].append(run['history']['train_loss'])
        convergence['sgd_stop_loss'].append(run['history']['stop_loss'])
        epochs_run, best_epoch = stopping_epoch(run['history']['stop_loss'])
        convergence['stop_epochs'].append(epochs_run)
        convergence['best_epochs'].append(best_epoch)
    # Consistency check: the early-stopped run must stop where the recorded curve says.
    _, check = train_sgd(Zc, yc, eta_single, SEED, Zstop=Zc_stop, ystop=yc_stop, early_stop=True)
    assert check['epochs_run'] == convergence['stop_epochs'][0]

    lbfgs_caps = [1, 2, 3, 5, 8, 12, 16, 20, 30, 40, 60, 80, 100]
    lbfgs_path = []
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        for cap in lbfgs_caps:
            m = LogisticRegression(max_iter=cap).fit(Zc, yc)
            used = int(np.atleast_1d(m.n_iter_).max())
            lbfgs_path.append({'iterations': used,
                               'train_loss': log_loss_from_scores(m.decision_function(Zc), yc),
                               'stop_loss': log_loss_from_scores(m.decision_function(Zc_stop), yc_stop)})
            if used < cap:
                break
    convergence['lbfgs_path'] = lbfgs_path

    # Stability of the final score: plain SGD on the full training rows, five seeds.
    seed_f1 = []
    for seed in STABILITY_SEEDS:
        fitted, _, _ = fit_config('sgd', Xtr, ytr, gtr, eta_single, seed)
        seed_f1.append(evaluate(fitted, Xva, yva)['f1'])
    convergence['sgd_validation_f1_by_seed'] = seed_f1

sgd_train = np.array(convergence['sgd_train_loss'])
lbfgs_final = lbfgs_path[-1]['train_loss']
target = lbfgs_final * 1.01
sgd_mean = sgd_train.mean(axis=0)
sgd_pass_to_target = next((i + 1 for i, v in enumerate(sgd_mean) if v <= target), None)
lbfgs_pass_to_target = next((p['iterations'] for p in lbfgs_path if p['train_loss'] <= target), None)
convergence.update(target_loss=target, sgd_passes_to_target=sgd_pass_to_target,
                   lbfgs_passes_to_target=lbfgs_pass_to_target)
print('Early stopping would stop at epoch (per seed):', convergence['stop_epochs'])
print('Best epoch (per seed):', convergence['best_epochs'])
print('SGD training loss after epochs 1, 5, 10, 20 (seed mean):',
      [round(float(sgd_mean[i]), 4) for i in [0, 4, 9, EPOCH_BUDGET - 1]])
print('SGD training loss spread across seeds at the last epoch:',
      round(float(sgd_train[:, -1].max() - sgd_train[:, -1].min()), 5))
print('L-BFGS final training loss:', round(lbfgs_final, 4), 'after', lbfgs_path[-1]['iterations'], 'iterations')
print('Passes to get within 1% of the L-BFGS final loss: SGD', sgd_pass_to_target,
      '| L-BFGS', lbfgs_pass_to_target)
print('Plain SGD validation F1 by seed:', [round(v, 4) for v in seed_f1],
      '| range', round(max(seed_f1) - min(seed_f1), 4))

## 6. Five-fold cross-validation

Grouped, stratified 5-fold cross-validation over the 24,000 training and validation
rows, for all four configurations. Inside every fold, the learning rate is tuned again
and the stopping set is carved out of that fold's training rows only. The evaluation
fold is used for scoring and nothing else.

In [ ]:
cv_folds = {name: [] for name in CONFIGS}
cv_eta = []
cv_seconds = {name: 0.0 for name in CONFIGS}
cv_tuning_seconds = 0.0
for fold_number, (fold_train, fold_eval) in enumerate(folds, start=1):
    Xf, yf, gf = Xtv.iloc[fold_train], ytv[fold_train], gtv[fold_train]
    Xe, ye = Xtv.iloc[fold_eval], ytv[fold_eval]
    start = time.perf_counter()
    with Spend('cross_validation'):
        eta_fold, _ = tune_eta(Xf, yf, gf, SEED + fold_number)
    cv_tuning_seconds += time.perf_counter() - start
    cv_eta.append(eta_fold)
    for name in CONFIGS:
        start = time.perf_counter()
        with Spend('cross_validation'):
            fitted, times, info = fit_config(name, Xf, yf, gf, eta_fold, SEED + fold_number)
            scores = evaluate(fitted, Xe, ye)
        cv_seconds[name] += time.perf_counter() - start
        cv_folds[name].append({**scores, 'fit_total_s': times['total'], 'epochs': info['epochs']})

cv_summary = {}
baseline_f1_folds = np.array([f['f1'] for f in cv_folds['baseline']])
for name in CONFIGS:
    frame = pd.DataFrame(cv_folds[name])
    stats = {metric: {'mean': float(frame[metric].mean()), 'std': float(frame[metric].std(ddof=1))}
             for metric in ['f1', 'recall', 'precision', 'accuracy', 'roc_auc', 'fit_total_s']}
    paired = frame['f1'].to_numpy() - baseline_f1_folds
    stats['f1_diff_vs_baseline'] = {'mean': float(paired.mean()), 'std': float(paired.std(ddof=1))}
    stats['f1_standard_error'] = stats['f1']['std'] / np.sqrt(len(frame))
    stats['mean_epochs'] = float(frame['epochs'].mean())
    stats['cv_wall_seconds'] = cv_seconds[name]
    stats['cv_cost_vs_single_fit'] = cv_seconds[name] / timing[name]['total']
    cv_summary[name] = stats

cv_table = pd.DataFrame([{
    'configuration': LABELS[n],
    'cv_f1_mean': cv_summary[n]['f1']['mean'], 'cv_f1_std': cv_summary[n]['f1']['std'],
    'cv_recall_mean': cv_summary[n]['recall']['mean'], 'cv_recall_std': cv_summary[n]['recall']['std'],
    'cv_accuracy_mean': cv_summary[n]['accuracy']['mean'],
    'cv_f1_diff_vs_baseline': cv_summary[n]['f1_diff_vs_baseline']['mean'],
    'single_split_f1': single[n]['validation']['f1'],
    'cv_wall_s': cv_summary[n]['cv_wall_seconds'],
    'cv_cost_vs_single_fit': cv_summary[n]['cv_cost_vs_single_fit']} for n in CONFIGS])
print('Learning rate chosen in each fold:', cv_eta)
print('Learning-rate tuning inside CV took', round(cv_tuning_seconds, 2), 'seconds in total.')
print(cv_table.round(4).to_string(index=False))

## 7. Model selection with the predefined rule

The rule from the top of the notebook is applied to the cross-validation results and
the median training times. Nothing here was adjusted after seeing the numbers.

In [ ]:
best_name = max(CONFIGS, key=lambda n: cv_summary[n]['f1']['mean'])
threshold = cv_summary[best_name]['f1']['mean'] - cv_summary[best_name]['f1_standard_error']
eligible = [n for n in CONFIGS if cv_summary[n]['f1']['mean'] >= threshold]
recommended = min(eligible, key=lambda n: timing[n]['total'])
not_chosen = [n for n in CONFIGS if n != recommended]
# The rejected alternative reported in the brief: a tested configuration that either
# failed the F1 condition or, if all passed, the slowest eligible one.
ineligible = [n for n in CONFIGS if n not in eligible]
rejected = (min(ineligible, key=lambda n: cv_summary[n]['f1']['mean']) if ineligible
            else max(not_chosen, key=lambda n: timing[n]['total']))
selection = {'best_f1_configuration': best_name, 'f1_threshold': threshold,
             'eligible': eligible, 'recommended': recommended, 'rejected_alternative': rejected,
             'rejected_reason': ('mean CV F1 below the one-standard-error threshold'
                                 if rejected in ineligible else
                                 'eligible on F1 but the slowest total training time')}
print('Best mean CV F1:', LABELS[best_name], round(cv_summary[best_name]['f1']['mean'], 4))
print('Eligibility threshold (best mean minus one standard error):', round(threshold, 4))
print('Eligible:', [LABELS[n] for n in eligible])
print('Recommended:', LABELS[recommended],
      f"| total {timing[recommended]['total']:.4f} s",
      f"| speedup vs baseline {timing['baseline']['total'] / timing[recommended]['total']:.2f}x",
      f"| CV F1 difference {cv_summary[recommended]['f1_diff_vs_baseline']['mean']:+.4f}")
print('Rejected alternative:', LABELS[rejected], '|', selection['rejected_reason'])

## 8. Runtime scaling

Total training time (preprocessing plus fit) at 25%, 50% and 100% of the training
rows. The slope of a straight line through log(time) against log(rows) estimates how
cost grows with data size. A slope near 1 means time grows in step with the row count.

In [ ]:
scaling_rows = []
rng = np.random.RandomState(SEED)
order = rng.permutation(len(ytr))
with Spend('scaling'):
    for fraction in SCALING_FRACTIONS:
        take = np.sort(order[:int(round(fraction * len(ytr)))])
        Xs, ys, gs = Xtr.iloc[take], ytr[take], gtr[take]
        for name in ['baseline', 'sgd']:
            t = time_config(name, Xs, ys, gs, eta_single, SEED, repeats=SCALING_REPEATS)
            scaling_rows.append({'configuration': name, 'fraction': fraction, 'rows': len(take),
                                 'preprocess_s': t['preprocess'], 'fit_s': t['fit'],
                                 'total_s': t['total']})
scaling_table = pd.DataFrame(scaling_rows)
scaling_slopes = {}
for name in ['baseline', 'sgd']:
    part = scaling_table[scaling_table.configuration == name]
    scaling_slopes[name] = {
        'total': float(np.polyfit(np.log(part.rows), np.log(part.total_s), 1)[0]),
        'fit': float(np.polyfit(np.log(part.rows), np.log(part.fit_s), 1)[0]),
        'preprocess': float(np.polyfit(np.log(part.rows), np.log(part.preprocess_s), 1)[0])}
print(scaling_table.round(4).to_string(index=False))
print('Log-log slopes:', {k: {kk: round(vv, 2) for kk, vv in v.items()} for k, v in scaling_slopes.items()})

## 9. Compute budget ledger

All fitting, tuning, cross-validation and timing work is counted against the
15-minute budget. Data loading and plotting are not model work and are reported
separately as the notebook's total run time.

In [ ]:
budget_used = float(sum(ledger.values()))
budget = {'budget_seconds': BUDGET_SECONDS, 'used_seconds': budget_used,
          'used_share': budget_used / BUDGET_SECONDS,
          'within_budget': budget_used <= BUDGET_SECONDS,
          'by_stage': {k: float(v) for k, v in ledger.items()}}
for stage, seconds in ledger.items():
    print(f'{stage:<20} {seconds:8.2f} s')
print(f"{'TOTAL':<20} {budget_used:8.2f} s of {BUDGET_SECONDS} s "
      f"({100 * budget['used_share']:.1f}% of the budget)")
assert budget['within_budget'], 'The experiment went over the fixed compute budget.'

## 10. Figures

Figure 1 is the convergence curve, Figure 2 the runtime comparison and Figure 3 the
performance comparison. Each is saved as a PNG in the output folder.

In [ ]:
# Figure 1: convergence
fig, ax = plt.subplots(figsize=(6.5, 3.8))
epochs = np.arange(1, EPOCH_BUDGET + 1)
sgd_stop = np.array(convergence['sgd_stop_loss'])
ax.fill_between(epochs, sgd_train.min(axis=0), sgd_train.max(axis=0),
                color=COLORS['sgd'], alpha=0.2, linewidth=0)
ax.plot(epochs, sgd_train.mean(axis=0), color=COLORS['sgd'], linewidth=2, marker='o',
        markersize=3.5, label='SGD training loss (mean of 5 seeds, band = range)')
ax.plot(epochs, sgd_stop.mean(axis=0), color=COLORS['sgd_early_stop'], linewidth=2,
        linestyle='--', label='SGD stopping-set loss (mean of 5 seeds)')
lb_x = [p['iterations'] for p in lbfgs_path]
lb_y = [p['train_loss'] for p in lbfgs_path]
ax.plot(lb_x, lb_y, color=COLORS['baseline'], linewidth=2, marker='s', markersize=3.5,
        label='Baseline L-BFGS training loss')
stop_at = int(np.median(convergence['stop_epochs']))
ax.axvline(stop_at, color=MUTED, linewidth=1, linestyle=':')
ymax = max(max(lb_y[1:] or lb_y), float(sgd_train.max()), float(sgd_stop.max()))
ymin = min(min(lb_y), float(sgd_train.min()), float(sgd_stop.min()))
pad = 0.08 * (ymax - ymin)
ax.set_ylim(ymin - pad, ymax + pad)
ax.text(stop_at * 0.93, ymin - 0.5 * pad, f'early stopping ends at epoch {stop_at} (median)',
        fontsize=8.5, color=MUTED, ha='right', va='center')
ax.set_xscale('log')
ax.set_xticks([1, 2, 5, 10, 20, 50, 100])
ax.set_xticklabels(['1', '2', '5', '10', '20', '50', '100'])
ax.set_xlabel('Passes over the training rows (SGD epochs or L-BFGS iterations, log scale)',
              color=INK)
ax.set_ylabel('Log loss', color=INK)
ax.grid(axis='y', color='#e4e3df', linewidth=0.7)
ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=8.5, loc='upper right')
fig.tight_layout()
fig.savefig(OUT / 'fig1_convergence.png', dpi=300)
plt.show()

In [ ]:
# Figure 2: runtime comparison, stacked by stage
fig, ax = plt.subplots(figsize=(6.5, 3.0))
names = [LABELS[n] for n in CONFIGS]
ypos = np.arange(len(CONFIGS))[::-1]
stage_colors = {'preprocess': '#86b6ef', 'extra': '#eda100', 'fit': '#184f95'}
stage_labels = {'preprocess': 'Preprocessing', 'extra': 'Selection or stopping split',
                'fit': 'Model fit'}
left = np.zeros(len(CONFIGS))
for stage in ['preprocess', 'extra', 'fit']:
    values = np.array([timing[n][stage] for n in CONFIGS])
    ax.barh(ypos, values, left=left, height=0.55, color=stage_colors[stage],
            edgecolor='white', linewidth=1.5, label=stage_labels[stage])
    left += values
for pos, name in zip(ypos, CONFIGS):
    speed = timing['baseline']['total'] / timing[name]['total']
    ax.text(timing[name]['total'] + 0.01 * left.max(), pos,
            f"{timing[name]['total']:.3f} s ({speed:.2f}x)", va='center', fontsize=9, color=INK)
ax.set_yticks(ypos)
ax.set_yticklabels(names)
ax.set_xlim(0, left.max() * 1.3)
ax.set_xlabel('Median training time (s); speedup vs. baseline in parentheses', color=INK)
ax.grid(axis='x', color='#e4e3df', linewidth=0.7)
ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=8.5, ncol=3, loc='lower center', bbox_to_anchor=(0.45, 1.0))
fig.tight_layout()
fig.savefig(OUT / 'fig2_runtime.png', dpi=300)
plt.show()

In [ ]:
# Figure 3: performance comparison from 5-fold cross-validation
fig, ax = plt.subplots(figsize=(6.5, 3.4))
xpos = np.arange(len(CONFIGS))
width = 0.36
metric_colors = {'f1': '#2a78d6', 'recall': '#eb6834'}
for offset, metric, label in [(-width / 2, 'f1', 'F1'), (width / 2, 'recall', 'Recall')]:
    means = [cv_summary[n][metric]['mean'] for n in CONFIGS]
    stds = [cv_summary[n][metric]['std'] for n in CONFIGS]
    bars = ax.bar(xpos + offset, means, width - 0.03, yerr=stds, color=metric_colors[metric],
                  ecolor=MUTED, capsize=3, error_kw={'linewidth': 1}, label=label)
    for rect, mean, std in zip(bars, means, stds):
        ax.text(rect.get_x() + rect.get_width() / 2, mean + std + 0.008, f'{mean:.3f}',
                ha='center', fontsize=8.5, color=INK)
ax.set_xticks(xpos)
ax.set_xticklabels([LABELS[n].replace(' + ', ' +\n') for n in CONFIGS], fontsize=9)
ax.set_ylabel('5-fold CV score (bar = mean, line = SD)', color=INK)
ax.set_ylim(0, max(cv_summary[n]['f1']['mean'] + cv_summary[n]['f1']['std'] for n in CONFIGS) + 0.12)
ax.grid(axis='y', color='#e4e3df', linewidth=0.7)
ax.set_axisbelow(True)
ax.legend(frameon=False, fontsize=9, ncol=2, loc='upper right')
fig.tight_layout()
fig.savefig(OUT / 'fig3_performance.png', dpi=300)
plt.show()

## 11. Save the results

`results.json` holds every number used in the report. The CSV files are the tables.

In [ ]:
results = {
    'meta': {
        'python': platform.python_version(), 'platform': platform.platform(),
        'processor': platform.processor(), 'cpu_count': os.cpu_count(), 'threads_used': 1,
        'versions': {p: version(p) for p in ['numpy', 'pandas', 'scikit-learn', 'scipy',
                                             'matplotlib', 'threadpoolctl']},
        'archive_sha256': archive_hash, 'split_source': split_source, 'seed': SEED,
        'rows': {'train': int(len(train_i)), 'validation': int(len(val_i)),
                 'test_not_used': int(len(test_i)), 'cv_rows': int(len(trainval_i))},
        'encoded_features': int(Z_baseline.shape[1]),
        'default_rate_train': float(ytr.mean()),
        'notebook_wall_seconds': time.perf_counter() - NOTEBOOK_START,
        'notebook_cpu_seconds': time.process_time() - CPU_START},
    'settings': {'epoch_budget': EPOCH_BUDGET, 'eta_grid': ETA_GRID, 'alpha': ALPHA,
                 'patience': PATIENCE, 'min_delta': MIN_DELTA, 'stop_fraction': STOP_FRACTION,
                 'k_features': K_FEATURES, 'timing_repeats': TIMING_REPEATS,
                 'budget_seconds': BUDGET_SECONDS},
    'baseline': {'lbfgs_iterations': single['baseline']['info']['epochs'],
                 'converged': bool(baseline_converged), 'train_log_loss': baseline_train_loss},
    'tuning': {'chosen_eta0': eta_single, 'table': eta_table, 'cv_eta0_by_fold': cv_eta,
               'cv_tuning_seconds': cv_tuning_seconds},
    'single_split': single,
    'timing': timing,
    'runtime_table': runtime_rows,
    'convergence': convergence,
    'cross_validation': {'folds': cv_folds, 'summary': cv_summary},
    'selection': selection,
    'scaling': {'rows': scaling_rows, 'slopes': scaling_slopes},
    'budget': budget,
}
(OUT / 'results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
runtime_table.to_csv(OUT / 'runtime_table.csv', index=False)
cv_table.to_csv(OUT / 'cv_table.csv', index=False)
scaling_table.to_csv(OUT / 'scaling_table.csv', index=False)
print('Files:', sorted(p.name for p in OUT.iterdir()))
print('Recommended configuration:', LABELS[recommended])
print(f"Budget used: {budget_used:.1f} s of {BUDGET_SECONDS} s. "
      f"Whole notebook: {results['meta']['notebook_wall_seconds']:.1f} s.")